# Amazon Redshift practical: load e-commerce CSV data from Amazon S3

This notebook is documentation-first and matches the style of D361_BasicRedshift: every command appears in a syntax-highlighted Markdown block. Run SQL in Redshift Query Editor v2 and Bash commands in a terminal with AWS CLI configured.

The practical creates an ecomm schema, uploads the included customer/order CSV files to S3, configures the IAM access path, loads with COPY, validates the results, and runs analytical queries.

> Use a provisioned Redshift cluster. Replace every value shown in angle brackets before execution. Keep the cluster and S3 bucket in the same AWS Region where practical.

## 1. Choose how to load the sample CSV files

Three options are available.

| Option | Best use | Requirements |
|---|---|---|
| A — included CSV files and AWS CLI | recommended repeatable practical | AWS CLI, S3 bucket, Redshift IAM role |
| B — Query Editor v2 Load data wizard | quick portal walkthrough | browser access and permission to create/load tables |
| C — your own customers/orders CSV | adapting the lab | headers and types matching the target tables |

Included files:

~~~text
D36_Redshift/data/customers.csv
D36_Redshift/data/orders.csv
D36_Redshift/data/order_items.csv
~~~

Recommended S3 layout:

~~~text
s3://<bucket-name>/redshift/ecomm/customers/customers.csv
s3://<bucket-name>/redshift/ecomm/orders/orders.csv
s3://<bucket-name>/redshift/ecomm/order_items/order_items.csv
~~~

COPY cannot read a file directly from a laptop. Upload it to S3 first, or use the Query Editor v2 Load data workflow.

## 2. Option A — upload the included files with AWS CLI

Run from the course repository root. These commands upload only the three named files and do not synchronize or delete other objects.

~~~bash
aws sts get-caller-identity --profile training
aws s3api head-bucket --bucket <bucket-name> --profile training

aws s3 cp D36_Redshift/data/customers.csv   s3://<bucket-name>/redshift/ecomm/customers/customers.csv   --region <aws-region> --profile training

aws s3 cp D36_Redshift/data/orders.csv   s3://<bucket-name>/redshift/ecomm/orders/orders.csv   --region <aws-region> --profile training

aws s3 cp D36_Redshift/data/order_items.csv   s3://<bucket-name>/redshift/ecomm/order_items/order_items.csv   --region <aws-region> --profile training

aws s3 ls s3://<bucket-name>/redshift/ecomm/   --recursive --region <aws-region> --profile training
~~~

The AWS CLI identity needs s3:PutObject for the destination prefix. That upload identity can be different from the IAM role assumed by Redshift during COPY.

## 3. Option B — Query Editor v2 Load data wizard

1. Open **Amazon Redshift → Query Editor v2**.
2. Connect to the provisioned cluster and select ecommdb.
3. Create the tables from this notebook before loading.
4. In the database explorer, select the target table and choose **Load data**.
5. Select **Load from S3**, enter the exact object URI, and select the cluster IAM role.
6. Choose CSV, comma delimiter, one header row, and automatic date/time parsing.
7. Verify column mapping before starting the load.
8. Open load details if a row is rejected.

The wizard generates a COPY operation. Preserve the generated SQL or use the explicit COPY commands later in this notebook for a repeatable pipeline.

## 4. The complete permission chain

All four layers must be correct:

~~~text
Deployment identity
  └─ iam:PassRole + redshift:ModifyClusterIamRoles
          ↓
Provisioned Redshift cluster
  └─ attached IAM role
          ↓ sts:AssumeRole allowed by trust policy
RedshiftS3ReadRole
  └─ s3:ListBucket + s3:GetObject
          ↓
S3 bucket and redshift/ecomm/* objects

Database user
  └─ USAGE + INSERT/SELECT + ASSUMEROLE for COPY
~~~

The Redshift IAM role reads S3. The database grants authorize SQL. The identity attaching the role needs iam:PassRole. These permissions are separate and none substitutes for another.

## 5. Create the Redshift-to-S3 IAM role

Create an IAM role named RedshiftS3ReadRole. Its trust policy allows the Redshift service to assume it.

**Trust policy**

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "redshift.amazonaws.com"
      },
      "Action": "sts:AssumeRole"
    }
  ]
}
~~~

**Scoped S3 read policy**

Replace the bucket name in both resources. ListBucket applies to the bucket ARN; GetObject applies to object ARNs.

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "ListEcommPrefix",
      "Effect": "Allow",
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::<bucket-name>",
      "Condition": {
        "StringLike": {
          "s3:prefix": [
            "redshift/ecomm",
            "redshift/ecomm/*"
          ]
        }
      }
    },
    {
      "Sid": "ReadEcommObjects",
      "Effect": "Allow",
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::<bucket-name>/redshift/ecomm/*"
    }
  ]
}
~~~

For a simple lab, the AWS-managed AmazonS3ReadOnlyAccess policy also works but grants read access to all S3 buckets visible to the account. The scoped policy is preferred.

## 6. Optional permissions: KMS and cross-account S3

If the objects use SSE-S3 encryption, no KMS permission is needed. For SSE-KMS, add permission for the exact customer-managed KMS key and ensure its key policy permits the role.

~~~json
{
  "Sid": "DecryptEcommFiles",
  "Effect": "Allow",
  "Action": [
    "kms:Decrypt"
  ],
  "Resource": "arn:aws:kms:<aws-region>:<account-id>:key/<key-id>"
}
~~~

If the bucket belongs to another AWS account, its bucket policy must also allow the Redshift role:

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "AllowRedshiftRoleRead",
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::<redshift-account-id>:role/RedshiftS3ReadRole"
      },
      "Action": [
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::<bucket-name>/redshift/ecomm/*"
    },
    {
      "Sid": "AllowRedshiftRoleList",
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::<redshift-account-id>:role/RedshiftS3ReadRole"
      },
      "Action": "s3:ListBucket",
      "Resource": "arn:aws:s3:::<bucket-name>",
      "Condition": {
        "StringLike": {
          "s3:prefix": [
            "redshift/ecomm",
            "redshift/ecomm/*"
          ]
        }
      }
    }
  ]
}
~~~

With enhanced VPC routing, the cluster also needs a working network path to S3, normally an S3 gateway endpoint or NAT route.

## 7. Permission needed by the identity attaching the role

The administrator or automation identity attaching the role needs iam:PassRole, restricted to Redshift, plus permission to modify the cluster IAM roles.

~~~json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "PassRedshiftS3Role",
      "Effect": "Allow",
      "Action": "iam:PassRole",
      "Resource": "arn:aws:iam::<account-id>:role/RedshiftS3ReadRole",
      "Condition": {
        "StringEquals": {
          "iam:PassedToService": "redshift.amazonaws.com"
        }
      }
    },
    {
      "Sid": "AttachRoleToTrainingCluster",
      "Effect": "Allow",
      "Action": [
        "redshift:ModifyClusterIamRoles",
        "redshift:DescribeClusters"
      ],
      "Resource": "*"
    }
  ]
}
~~~

iam:PassRole belongs to the caller attaching or selecting the role. It should not be added to the S3-read policy merely to make COPY work.

## 8. Attach the role to the provisioned cluster

**Console method**

1. Open **Amazon Redshift → Provisioned clusters**.
2. Select the cluster.
3. Open **Properties → Cluster permissions → Manage IAM roles**.
4. Associate RedshiftS3ReadRole.
5. Mark it as the default role if IAM_ROLE default will be used.
6. Save and wait until the cluster modification is complete.

**AWS CLI method**

~~~bash
aws redshift modify-cluster-iam-roles   --cluster-identifier <cluster-identifier>   --add-iam-roles arn:aws:iam::<account-id>:role/RedshiftS3ReadRole   --default-iam-role-arn arn:aws:iam::<account-id>:role/RedshiftS3ReadRole   --region <aws-region>   --profile training

aws redshift describe-clusters   --cluster-identifier <cluster-identifier>   --query "Clusters[0].{Status:ClusterStatus,Roles:IamRoles,DefaultRole:DefaultIamRoleArn}"   --region <aws-region>   --profile training
~~~

Wait for Status to return available before loading.

## 9. Create and connect to ecommdb

Connect to an existing database such as dev. CREATE DATABASE must run outside an explicit transaction.

~~~sql
CREATE DATABASE ecommdb;
~~~

Then create a new Query Editor connection to ecommdb and verify it:

~~~sql
SELECT current_database() AS database_name,
       current_user AS user_name,
       GETDATE() AS cluster_time;
~~~

## 10. Create schema and target tables

The tables use Redshift distribution and sort choices suitable for the expected joins. Primary and foreign keys are informational and must describe valid data.

~~~sql
CREATE SCHEMA IF NOT EXISTS ecomm;

CREATE TABLE IF NOT EXISTS ecomm.customers (
    customer_id      BIGINT       NOT NULL,
    customer_name    VARCHAR(100) NOT NULL,
    email            VARCHAR(200),
    city             VARCHAR(80),
    state_code       CHAR(2),
    signup_date      DATE,
    PRIMARY KEY (customer_id)
)
DISTSTYLE ALL
SORTKEY (customer_id);

CREATE TABLE IF NOT EXISTS ecomm.orders (
    order_id          BIGINT        NOT NULL,
    customer_id       BIGINT        NOT NULL,
    order_timestamp   TIMESTAMP     NOT NULL,
    order_status      VARCHAR(20)   NOT NULL,
    shipping_amount   DECIMAL(12,2),
    PRIMARY KEY (order_id),
    FOREIGN KEY (customer_id) REFERENCES ecomm.customers(customer_id)
)
DISTSTYLE KEY
DISTKEY (order_id)
COMPOUND SORTKEY (order_timestamp, order_id);

CREATE TABLE IF NOT EXISTS ecomm.order_items (
    order_id          BIGINT        NOT NULL,
    line_number       SMALLINT      NOT NULL,
    product_id        BIGINT        NOT NULL,
    quantity          INTEGER       NOT NULL,
    unit_price        DECIMAL(12,2) NOT NULL,
    discount_amount   DECIMAL(12,2),
    PRIMARY KEY (order_id, line_number),
    FOREIGN KEY (order_id) REFERENCES ecomm.orders(order_id)
)
DISTSTYLE KEY
DISTKEY (order_id)
COMPOUND SORTKEY (order_id, product_id);
~~~

## 11. Grant database permissions to a loader user or group

IAM authorizes access to S3; these grants authorize actions inside Redshift. Replace the principal. Redshift supports users and groups; role-based access control can also be used where enabled.

~~~sql
-- Run as the schema/table owner or an administrator.
GRANT USAGE ON SCHEMA ecomm TO <loader_user>;
GRANT INSERT, SELECT ON TABLE ecomm.customers TO <loader_user>;
GRANT INSERT, SELECT ON TABLE ecomm.orders TO <loader_user>;
GRANT INSERT, SELECT ON TABLE ecomm.order_items TO <loader_user>;

-- Run as a database superuser when fine-grained IAM-role access is enabled.
GRANT ASSUMEROLE
ON 'arn:aws:iam::<account-id>:role/RedshiftS3ReadRole'
TO <loader_user>
FOR COPY;
~~~

A superuser always retains ASSUMEROLE. To enforce fine-grained IAM-role access cluster-wide, a superuser can first run `REVOKE ASSUMEROLE ON ALL FROM PUBLIC FOR ALL`; review existing workloads before doing this because it changes access for every database user. If the loader must create tables, grant that separately and only when required:

~~~sql
GRANT CREATE ON SCHEMA ecomm TO <loader_user>;
-- ALTER and DROP authority follow object ownership; do not grant broad ownership casually.
~~~

## 12. Validate CSV parsing with COPY NOLOAD

NOLOAD parses and type-checks files without inserting rows. Use either IAM_ROLE default after setting the cluster default role, or specify the exact attached role ARN.

~~~sql
COPY ecomm.customers
FROM 's3://<bucket-name>/redshift/ecomm/customers/customers.csv'
IAM_ROLE default
REGION '<aws-region>'
FORMAT AS CSV
IGNOREHEADER 1
EMPTYASNULL
BLANKSASNULL
DATEFORMAT 'auto'
TIMEFORMAT 'auto'
NOLOAD;

COPY ecomm.orders
FROM 's3://<bucket-name>/redshift/ecomm/orders/orders.csv'
IAM_ROLE 'arn:aws:iam::<account-id>:role/RedshiftS3ReadRole'
REGION '<aws-region>'
FORMAT AS CSV
IGNOREHEADER 1
EMPTYASNULL
BLANKSASNULL
DATEFORMAT 'auto'
TIMEFORMAT 'auto'
NOLOAD;
~~~

Using the explicit ARN makes the dependency visible. IAM_ROLE default is shorter and uses the role marked as default on the cluster.

## 13. Perform a repeatable full load

TRUNCATE removes existing rows from only these three lab tables. Load parent tables before child tables even though Redshift does not enforce foreign keys.

~~~sql
TRUNCATE TABLE ecomm.order_items;
TRUNCATE TABLE ecomm.orders;
TRUNCATE TABLE ecomm.customers;

COPY ecomm.customers
FROM 's3://<bucket-name>/redshift/ecomm/customers/customers.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
DATEFORMAT 'auto' TIMEFORMAT 'auto'
COMPUPDATE ON STATUPDATE ON;

COPY ecomm.orders
FROM 's3://<bucket-name>/redshift/ecomm/orders/orders.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
DATEFORMAT 'auto' TIMEFORMAT 'auto'
COMPUPDATE ON STATUPDATE ON;

COPY ecomm.order_items
FROM 's3://<bucket-name>/redshift/ecomm/order_items/order_items.csv'
IAM_ROLE default REGION '<aws-region>'
CSV IGNOREHEADER 1 EMPTYASNULL BLANKSASNULL
DATEFORMAT 'auto' TIMEFORMAT 'auto'
COMPUPDATE ON STATUPDATE ON;
~~~

## 14. Validate the load

Expected counts are 5 customers, 6 orders, and 8 order items.

~~~sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM ecomm.customers
UNION ALL
SELECT 'orders', COUNT(*) FROM ecomm.orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM ecomm.order_items
ORDER BY table_name;

SELECT 'duplicate customers' AS check_name,
       COUNT(*) - COUNT(DISTINCT customer_id) AS violations
FROM ecomm.customers
UNION ALL
SELECT 'duplicate orders',
       COUNT(*) - COUNT(DISTINCT order_id)
FROM ecomm.orders
UNION ALL
SELECT 'orders without customer', COUNT(*)
FROM ecomm.orders o
LEFT JOIN ecomm.customers c ON c.customer_id = o.customer_id
WHERE c.customer_id IS NULL
UNION ALL
SELECT 'items without order', COUNT(*)
FROM ecomm.order_items i
LEFT JOIN ecomm.orders o ON o.order_id = i.order_id
WHERE o.order_id IS NULL;
~~~

Every violation count should be zero before relying on the declared constraints.

## 15. Troubleshoot COPY failures

SYS_LOAD_ERROR_DETAIL works on provisioned Redshift and exposes parsing errors.

~~~sql
SELECT query_id,
       start_time,
       TRIM(file_name) AS file_name,
       line_number,
       TRIM(column_name) AS column_name,
       TRIM(column_type) AS column_type,
       TRIM(error_message) AS error_message
FROM sys_load_error_detail
ORDER BY start_time DESC
LIMIT 50;
~~~

Review recent COPY operations:

~~~sql
SELECT query_id,
       status,
       start_time,
       end_time,
       loaded_rows,
       loaded_bytes,
       source_file_count,
       file_format
FROM sys_load_history
WHERE start_time >= DATEADD(hour, -2, GETDATE())
ORDER BY start_time DESC;
~~~

Common failures:

- AccessDenied: role is unattached, trust is wrong, S3/KMS policy is missing, or a bucket policy denies it.
- S3ServiceException redirect/region error: REGION does not match the bucket.
- Invalid digit/date: source field order does not match table columns.
- Load into table failed: inspect SYS_LOAD_ERROR_DETAIL immediately.
- Zero files: URI/prefix is wrong or the role lacks ListBucket.

## 16. Query the imported data

~~~sql
SELECT
    o.order_id,
    c.customer_name,
    o.order_timestamp,
    o.order_status,
    SUM(i.quantity * i.unit_price - COALESCE(i.discount_amount, 0)) AS merchandise_amount,
    MAX(COALESCE(o.shipping_amount, 0)) AS shipping_amount,
    SUM(i.quantity * i.unit_price - COALESCE(i.discount_amount, 0))
      + MAX(COALESCE(o.shipping_amount, 0)) AS order_total
FROM ecomm.orders o
JOIN ecomm.customers c ON c.customer_id = o.customer_id
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_status <> 'cancelled'
GROUP BY o.order_id, c.customer_name, o.order_timestamp, o.order_status
ORDER BY o.order_timestamp, o.order_id;
~~~

~~~sql
SELECT
    DATE_TRUNC('month', o.order_timestamp)::DATE AS order_month,
    COUNT(DISTINCT o.order_id) AS orders,
    SUM(i.quantity) AS units,
    SUM(i.quantity * i.unit_price - COALESCE(i.discount_amount, 0)) AS merchandise_revenue
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_status <> 'cancelled'
GROUP BY 1
ORDER BY 1;
~~~

## 17. Inspect physical design and the query plan

~~~sql
SELECT "schema", "table", diststyle, sortkey1,
       size AS size_mb, tbl_rows, skew_rows, unsorted, stats_off
FROM svv_table_info
WHERE "schema" = 'ecomm'
ORDER BY size DESC, "table";

EXPLAIN
SELECT o.order_id,
       SUM(i.quantity * i.unit_price - COALESCE(i.discount_amount, 0)) AS amount
FROM ecomm.orders o
JOIN ecomm.order_items i ON i.order_id = o.order_id
WHERE o.order_timestamp >= TIMESTAMP '2026-08-01 00:00:00'
  AND o.order_timestamp <  TIMESTAMP '2026-09-01 00:00:00'
GROUP BY o.order_id;
~~~

Because orders and order_items share DISTKEY(order_id), the large join is eligible for co-location. The date predicate matches the leading orders sort key for zone-map pruning on realistic data volumes.

## 18. Production loading considerations

- Prefer multiple similarly sized files so slices can load in parallel.
- Use a manifest when the exact file set matters.
- Keep bucket and cluster regions aligned when possible.
- Use IAM roles, never access keys inside COPY.
- Land into staging tables, validate, then MERGE into curated targets.
- Capture source filename and ingestion timestamp when lineage matters.
- Use COMPUPDATE and STATUPDATE intentionally; automatic optimization handles much routine work.
- Do not mix the same S3 files between scheduled COPY and another ingestion mechanism.
- Restrict the S3 prefix, KMS key, role attachment, and Redshift database grants.
- Monitor SYS_LOAD_HISTORY, SYS_LOAD_ERROR_DETAIL, and SYS_QUERY_HISTORY.

## 19. Cleanup

Nothing here deletes resources automatically.

~~~sql
-- Removes only the lab schema and all objects inside it.
-- DROP SCHEMA ecomm CASCADE;
~~~

Run S3 cleanup only after verifying the exact prefix:

~~~bash
aws s3 rm s3://<bucket-name>/redshift/ecomm/   --recursive --dryrun --region <aws-region> --profile training

# Remove --dryrun only after reviewing every listed object.
~~~

Detach the IAM role only when no other Redshift load or unload uses it:

~~~bash
aws redshift modify-cluster-iam-roles   --cluster-identifier <cluster-identifier>   --remove-iam-roles arn:aws:iam::<account-id>:role/RedshiftS3ReadRole   --region <aws-region> --profile training
~~~